In [0]:
%pip install --find-links /Volumes/nextgen_genie/default/libs --no-index \
    langchain==1.2.11 \
    langchain-core==1.2.18 \
    langchain-community==0.4.1 \
    langchain-experimental==0.3.4 \
    databricks-langchain==0.17.0 \
    databricks-sdk==0.97.0 \
    pandas==2.3.3 \
    tabulate==0.9.0 \
    mlflow-skinny==3.10.1 \
    mlflow[databricks]==3.10.1

dbutils.library.restartPython()


In [0]:
# ── Cell 2: Configuration ────────────────────────────────────────────────────
from mlflow.tracking import MlflowClient

AGENT_NAME      = "students_csv_agent"
ENDPOINT_NAME   = "students-csv-agent-endpoint"
DATABRICKS_HOST = "https://adb-7405616168141157.17.azuredatabricks.net"

# ⚠️ SECURITY: Replace with dbutils.secrets in production
# TOKEN = dbutils.secrets.get(scope="your-scope", key="databricks-pat")
TOKEN           = "YOUR_DATABRICKS_TOKEN_HERE"

UC_MODEL_NAME   = f"nextgen_genie.default.{AGENT_NAME}"   # <catalog>.<schema>.<model>

# Path to students CSV in Unity Catalog Volume
CSV_UC_PATH     = "/Volumes/nextgen_genie/default/data/students_data.csv"

print(f"Agent      : {AGENT_NAME}")
print(f"UC Model   : {UC_MODEL_NAME}")
print(f"Endpoint   : {ENDPOINT_NAME}")
print(f"CSV Path   : {CSV_UC_PATH}")


In [0]:
# ── Cell 3: Upload students_data.csv to UC Volume ────────────────────────────
# Run this once to place the CSV where the serving container can read it.
# Skip if already uploaded.
import os

LOCAL_CSV  = "/Volumes/nextgen_genie/default/data/students_data.csv"
SOURCE_CSV = "/path/to/students_data.csv"   # ← update or use dbutils.fs.cp

# If running interactively on the cluster, copy from DBFS / local:
# dbutils.fs.cp("dbfs:/FileStore/students_data.csv", f"file:{LOCAL_CSV}")

# Quick sanity check
import pandas as pd
df_check = pd.read_csv(LOCAL_CSV)
print(f"CSV loaded OK — shape: {df_check.shape}")
print(df_check.head(3).to_string())


In [0]:
# ── Cell 4: Define StudentsCsvAgent ─────────────────────────────────────────
import mlflow
import pandas as pd

class StudentsCsvAgent(mlflow.pyfunc.PythonModel):
    """
    MLflow PythonModel that wraps a LangChain Pandas DataFrame Agent.
    The agent runs inside an MLflow Experiment so every invocation is
    automatically traced and logged.

    Input  schema : { "query": str }
    Output schema : { "response": str }
    """

    # ------------------------------------------------------------------
    # load_context  – called ONCE when the container / serving replica starts
    # ------------------------------------------------------------------
    def load_context(self, context):
        import os
        import pandas as pd
        import mlflow
        from databricks_langchain import ChatDatabricks
        from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
        from langchain.agents.agent_types import AgentType

        # ── 1. LLM ────────────────────────────────────────────────────
        self.llm = ChatDatabricks(
            endpoint="databricks-meta-llama-3-3-70b-instruct",
            max_tokens=1024,
            temperature=0.0,
        )

        # ── 2. Load CSV ───────────────────────────────────────────────
        csv_path = os.getenv("STUDENTS_CSV_PATH", "/Volumes/nextgen_genie/default/data/students_data.csv")
        self.df = pd.read_csv(csv_path)

        # ── 3. Build Pandas Agent ─────────────────────────────────────
        self.agent = create_pandas_dataframe_agent(
            llm=self.llm,
            df=self.df,
            agent_type=AgentType.OPENAI_FUNCTIONS,   # works with ChatDatabricks tool-calling
            verbose=True,
            allow_dangerous_code=True,               # required by langchain-experimental
            prefix=(
                "You are a data analyst assistant. "
                "The dataframe `df` contains student records with columns: "
                "Name, Age, Subject, Grade, Teacher Feedback, Sports. "
                "Sports may contain comma-separated values. "
                "Always answer in clear, concise language."
            ),
        )

        # ── 4. MLflow Experiment for tracing ──────────────────────────
        mlflow.set_experiment("/Shared/students_csv_agent_experiment")
        print("StudentsCsvAgent loaded ✅")

    # ------------------------------------------------------------------
    # predict  – called on every request
    # ------------------------------------------------------------------
    def predict(self, context, model_input: pd.DataFrame, params=None) -> pd.DataFrame:
        import mlflow

        # ── Extract query string ──────────────────────────────────────
        query = None
        if isinstance(model_input, pd.DataFrame):
            if "query" in model_input.columns:
                query = model_input["query"].iloc[0]
            else:
                query = str(model_input.iloc[0, 0])
        elif isinstance(model_input, dict):
            query = model_input.get("query") or str(next(iter(model_input.values())))
        elif isinstance(model_input, list):
            first = model_input[0]
            query = first.get("query") if isinstance(first, dict) else str(first)
        else:
            query = str(model_input)

        # ── Run agent inside an MLflow Experiment run ─────────────────
        with mlflow.start_run(run_name="csv_agent_query", nested=True):
            mlflow.log_param("query", query[:250])   # truncate for MLflow param limit

            try:
                result = self.agent.invoke({"input": query})
                answer = result.get("output", str(result))
            except Exception as exc:
                answer = f"Agent error: {exc}"

            mlflow.log_metric("success", 1 if not answer.startswith("Agent error") else 0)
            mlflow.log_text(answer, "agent_response.txt")

        return pd.DataFrame([{"response": answer}])


print("StudentsCsvAgent class defined ✅")


In [0]:
# ── Cell 5: Log & Register to Unity Catalog ─────────────────────────────────
import mlflow
import pandas as pd
import glob
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import ColSpec, Schema

input_schema  = Schema([ColSpec("string", "query")])
output_schema = Schema([ColSpec("string", "response")])
signature     = ModelSignature(inputs=input_schema, outputs=output_schema)

# Bundle all wheels from the volume
whl_dir    = "/Volumes/nextgen_genie/default/libs"
code_paths = sorted(glob.glob(f"{whl_dir}/*.whl"))
print(f"Bundling {len(code_paths)} wheel(s) from {whl_dir}")

pip_reqs = [
    "--find-links code/",
    "--no-index",
    "mlflow==3.10.1",
    "langchain==1.2.11",
    "langchain-core==1.2.18",
    "langchain-community==0.4.1",
    "langchain-experimental==0.3.4",   # ← pandas agent
    "databricks-langchain==0.17.0",
    "databricks-sdk==0.97.0",
    "pandas==2.3.3",
    "tabulate==0.9.0",
]

input_example = pd.DataFrame([{"query": "How many students play Cricket?"}])

mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_name="register-students-csv-agent") as run:
    model_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model=StudentsCsvAgent(),
        signature=signature,
        code_paths=code_paths,
        pip_requirements=pip_reqs,
        registered_model_name=UC_MODEL_NAME,
        input_example=input_example,
        # Pass CSV path to serving container via environment variable
        model_config={"STUDENTS_CSV_PATH": CSV_UC_PATH},
    )
    print(f"Run ID        : {run.info.run_id}")
    print(f"Model URI     : {model_info.model_uri}")
    print(f"Registered as : {UC_MODEL_NAME}")

client = MlflowClient()
latest_version = max(
    int(mv.version)
    for mv in client.search_model_versions(f"name='{UC_MODEL_NAME}'")
)
print(f"Latest version: {latest_version}")


In [0]:
# ── Cell 6: Local Validation ─────────────────────────────────────────────────
from mlflow.models import validate_serving_input, convert_input_example_to_serving_input

model_uri = f"models:/{UC_MODEL_NAME}/{latest_version}"

serving_input = convert_input_example_to_serving_input(
    pd.DataFrame([{"query": "How many students play Cricket?"}])
)
print("Serving payload:")
print(serving_input)

print("\nValidating model locally...")
try:
    validate_serving_input(model_uri, serving_input)
    print("\n✅ Validation passed — safe to deploy")
except Exception as e:
    print(f"\n❌ Validation FAILED — do NOT deploy")
    print(f"   Error: {e}")
    raise


In [0]:
# ── Cell 7: Local predict() smoke test ──────────────────────────────────────
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Sample queries covering different aspects of the CSV
test_queries = [
    "How many students are there in total?",
    "What is the average age of students?",
    "Which subject has the most students?",
    "How many students play Cricket?",
    "List the top 5 students with grade A+ in Mathematics.",
    "Which student plays the most sports?",
]

print("Running local predict() tests...\n")
for q in test_queries:
    test_input = pd.DataFrame([{"query": q}])
    try:
        result = loaded_model.predict(test_input)
        print(f"Q: {q}")
        print(f"A: {result['response'].iloc[0]}")
        print("-" * 60)
    except Exception as e:
        print(f"Q: {q}")
        print(f"❌ Error: {e}")
        print("-" * 60)


In [0]:
# ── Cell 8: Deploy to Serving Endpoint ──────────────────────────────────────
import requests, time, json

headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

endpoint_config = {
    "name": ENDPOINT_NAME,
    "config": {
        "served_entities": [
            {
                "entity_name": UC_MODEL_NAME,
                "entity_version": str(latest_version),
                "workload_size": "Small",
                "scale_to_zero_enabled": True,
                "environment_vars": {
                    "DATABRICKS_HOST": DATABRICKS_HOST,
                    "DATABRICKS_TOKEN": TOKEN,
                    # Make CSV path available inside the container
                    "STUDENTS_CSV_PATH": CSV_UC_PATH,
                },
            }
        ],
        "traffic_config": {
            "routes": [
                {
                    "served_model_name": f"{AGENT_NAME}-{latest_version}",
                    "traffic_percentage": 100,
                }
            ]
        },
    },
}

create_resp = requests.post(
    f"{DATABRICKS_HOST}/api/2.0/serving-endpoints",
    headers=headers,
    json=endpoint_config,
)

if create_resp.status_code == 200:
    print(f"Endpoint '{ENDPOINT_NAME}' created ✅")
elif "already exists" in create_resp.text:
    for attempt in range(20):
        status_resp = requests.get(
            f"{DATABRICKS_HOST}/api/2.0/serving-endpoints/{ENDPOINT_NAME}",
            headers=headers,
        ).json()
        config_update = status_resp.get("state", {}).get("config_update", "")
        if config_update not in ("IN_PROGRESS", "NOT_READY"):
            break
        print(f"  Config update in progress … waiting 30s (attempt {attempt+1}/20)")
        time.sleep(30)

    update_resp = requests.put(
        f"{DATABRICKS_HOST}/api/2.0/serving-endpoints/{ENDPOINT_NAME}/config",
        headers=headers,
        json=endpoint_config["config"],
    )
    update_resp.raise_for_status()
    print(f"Endpoint '{ENDPOINT_NAME}' updated to version {latest_version} ✅")
else:
    print(f"Error: {create_resp.status_code} — {create_resp.text}")
    create_resp.raise_for_status()


In [0]:
# ── Cell 9: Endpoint Smoke Test ──────────────────────────────────────────────
import json
from mlflow.deployments import get_deploy_client

deploy_client = get_deploy_client("databricks")

smoke_queries = [
    "How many students are in the dataset?",
    "What grade distribution exists across all students?",
    "Which sports are most popular among students?",
]

print(f"Smoke-testing endpoint '{ENDPOINT_NAME}' ...\n")
for q in smoke_queries:
    payload = {"dataframe_records": [{"query": q}]}
    response = deploy_client.predict(endpoint=ENDPOINT_NAME, inputs=payload)
    answer = (
        response["predictions"][0]["response"]
        if isinstance(response, dict) and "predictions" in response
        else str(response)
    )
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 60)
